# Tutorial 12: Advanced LangChain Techniques

In this tutorial, we'll explore advanced LangChain techniques, including custom chain development, advanced prompt engineering, retrieval-augmented generation (RAG), and fine-tuning language models for specific tasks.

## Setup

First, let's import the necessary libraries and set up our environment:

In [1]:
import os
from typing import List, Dict, Any
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from pydantic import BaseModel, Field

# Initialize Groq LLM
llm = ChatGroq(
        model_name="llama-3.1-8b-instant",
        temperature=0.7,
        model_kwargs={"top_p": 0.8, "seed": 1337}
    )

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


## 1. Custom chain development

Let's create a custom chain that processes text through multiple steps:

In [2]:
# Custom text processing using a plain Python class (LCEL-compatible)
from langchain_core.runnables import RunnableLambda

class TextProcessingChain:
    """Simple text preprocessor — no LLM needed."""
    def invoke(self, inputs):
        text = inputs.get("input_text", inputs) if isinstance(inputs, dict) else inputs
        text = text.lower()
        text = ''.join(c for c in text if c.isalnum() or c.isspace())
        text = ' '.join(text.split())
        return {"processed_text": text}

text_processor = TextProcessingChain()
result = text_processor.invoke({"input_text": "Hello, World! This is a   TEST."})
print(result["processed_text"])

hello world this is a test


## 2. Prompt templating and management

Let's explore advanced prompt templating techniques:

In [3]:
# Define a complex prompt template
complex_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant specializing in {domain}."),
    ("human", "I need help with the following task: {task}"),
    ("ai", "Certainly! I'd be happy to help you with that. Could you provide more details about {detail_request}?"),
    ("human", "Here's more information: {additional_info}"),
    ("ai", "Thank you for the additional information. Based on what you've told me, here's my response:"),
])

# Create a chain using the complex prompt
# LCEL chain (replaces deprecated LLMChain)
complex_chain = complex_prompt | llm | StrOutputParser()

result = complex_chain.invoke({
    "domain": "software development",
    "task": "optimizing a slow database query",
    "detail_request": "the current query structure and database schema",
    "additional_info": "The query is a JOIN operation across three tables with multiple WHERE clauses. The database is PostgreSQL."
})

print(result)

 

To optimize the JOIN operation in PostgreSQL, I'll suggest the following steps:

1. **Examine the query structure**: Provide the query itself, so I can understand the specific JOIN operations, WHERE clauses, and any subqueries involved.
2. **Analyze the database schema**: Knowing the structure of the tables involved will help me identify potential performance bottlenecks.
3. **Check indexing**: Verify that the columns used in the WHERE clauses and JOIN conditions are properly indexed.
4. **Optimize JOIN order**: Reorder the tables in the JOIN operation to reduce the number of rows being joined and to take advantage of any existing indexes.
5. **Use efficient JOIN types**: Ensure that the correct JOIN type is being used (e.g., INNER JOIN, LEFT JOIN, etc.).
6. **Avoid using SELECT \***: Only select the columns that are necessary for the query, as retrieving unnecessary columns can slow down the query.
7. **Consider rewriting the query**: If the query is complex, it may be beneficial t

## 3. Implementing retrieval-augmented generation (RAG)

Let's implement a simple RAG system using FAISS and HuggingFace embeddings:

In [4]:
# Load and preprocess the document
loader = TextLoader("sample_document.txt")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)

# Create embeddings and vector store
try:
    embeddings = OllamaEmbeddings(model="all-minilm", base_url=os.getenv('OLLAMA_EMBEDDING_URL'))
    db = FAISS.from_documents(texts, embeddings)
except Exception:
    from langchain_community.embeddings import FakeEmbeddings
    embeddings = FakeEmbeddings(size=384)
    db = FAISS.from_documents(texts, embeddings)
    print("Ollama not available — using FakeEmbeddings for demo")

# Define the RAG prompt
rag_prompt = PromptTemplate(
    template="""Use the following pieces of context to answer the question at the end. 
If you don't know the answer, just say that you don't know, don't try to make up an answer.

{context}

Question: {question}
Answer: """,
    input_variables=["context", "question"]
)

def retrieve_and_generate(query: str) -> str:
    # Retrieve relevant documents
    docs = db.similarity_search(query)
    context = "\n".join([doc.page_content for doc in docs])
    
    # Generate answer using LCEL
    rag_chain = rag_prompt | llm | StrOutputParser()
    return rag_chain.invoke({"context": context, "question": query})

# Test the RAG system
result = retrieve_and_generate("What is the capital of France?")
print(result)

Ollama not available — using FakeEmbeddings for demo


Paris is the capital of France.


## 4. Fine-tuning language models for specific tasks

While we can't directly fine-tune the Groq model, we can simulate fine-tuning by creating a specialized prompt that adapts the model's behavior for a specific task. Let's create a 'fine-tuned' model for sentiment analysis:

In [5]:
sentiment_prompt = PromptTemplate(
    template="""You are a sentiment analysis expert. Your task is to analyze the sentiment of the given text and classify it as positive, negative, or neutral. 
    Provide a brief explanation for your classification.

    Text: {input_text}

    Sentiment: """,
    input_variables=["input_text"]
)

sentiment_chain = sentiment_prompt | llm | StrOutputParser()

def analyze_sentiment(text: str) -> str:
    return sentiment_chain.invoke({"input_text": text})

# Test the 'fine-tuned' sentiment analysis model
texts = [
    "I love this product! It's amazing and works perfectly.",
    "This is the worst experience I've ever had. Terrible customer service.",
    "The weather is quite nice today, not too hot or cold."
]

for text in texts:
    print(f"Text: {text}")
    print(f"Sentiment: {analyze_sentiment(text)}\n")

Text: I love this product! It's amazing and works perfectly.


Sentiment: Sentiment: Positive

Explanation: The text expresses strong enthusiasm and admiration for the product, using words like "love" and "amazing." The phrase "works perfectly" further reinforces the positive sentiment, indicating satisfaction with the product's performance. The overall tone is extremely positive, making it clear that the sentiment is positive.

Text: This is the worst experience I've ever had. Terrible customer service.


Sentiment: Sentiment: Negative

Explanation: The text contains two strong negative words: "worst" and "terrible". These words convey a sense of extreme dissatisfaction and frustration, indicating a strongly negative sentiment towards the experience and the customer service received. The overall tone of the text is one of anger and disappointment, which further supports the classification as negative.

Text: The weather is quite nice today, not too hot or cold.


Sentiment: Sentiment: Positive

Explanation: The text expresses a positive sentiment towards the weather, using the phrase "quite nice" to describe it. This phrase implies a pleasant and favorable atmosphere, which contributes to the overall positive tone of the text. There is no negative language or complaints, further supporting the classification as positive.



## Conclusion

In this tutorial, we've explored advanced LangChain techniques, including:

1. Custom chain development for specialized text processing
2. Advanced prompt templating and management for complex interactions
3. Implementing a retrieval-augmented generation (RAG) system
4. Simulating fine-tuning for specific tasks using specialized prompts

These techniques allow you to create more sophisticated and tailored AI applications using LangChain and LangGraph.

## Next Steps

To further advance your skills with LangChain and LangGraph, consider:

1. Experimenting with more complex custom chains and combining them with LangGraph flows
2. Developing a prompt management system for large-scale applications
3. Exploring advanced RAG techniques, such as hypothetical document embeddings or multi-query retrieval
4. Investigating ways to evaluate and improve the performance of your 'fine-tuned' models
5. Integrating these advanced techniques into a full-fledged application